In [18]:
# importing libraries for p1
import pandas as pd
import duckdb

In [19]:
test_query = """

SELECT *
FROM '/flutterwave_synthetic_txs.csv'
ORDER BY timestamp
LIMIT 05;

-- Other EDA checks using SQL

/*
SELECT DISTINCT card_country
FROM '/flutterwave_synthetic_txs.csv'


SELECT DISTINCT merchant_category
FROM '/flutterwave_synthetic_txs.csv'
*/

;

"""
display(duckdb.query(test_query).df())

,transaction_id,user_id,timestamp,amount_usd,card_country,merchant_category,device_id,dispute_status,is_fraud
0,460ef966-d7c8-40f8-b087-c2eb2c3181c5,USR_7223,2026-05-01 00:00:39+00:00,17.58,NG,ecommerce,DEV_36255,none,0
1,f9b36ab3-4b00-42b0-8cb6-14ee039bbd15,USR_4643,2026-05-01 00:01:25+00:00,18.41,NG,utility,DEV_36285,none,0
2,41685b8a-5ae8-46dd-b200-1e86241267e1,USR_1183,2026-05-01 00:01:53+00:00,55.73,GB,digital_goods,DEV_84459,none,0
3,500a1f97-d7fb-45dd-97bb-7839f565012c,USR_4727,2026-05-01 00:02:06+00:00,176.06,KE,utility,DEV_21148,none,0
4,b0873b43-3a43-4610-a4da-22640950543f,USR_1944,2026-05-01 00:02:06+00:00,240.15,US,utility,DEV_79367,none,0


In [20]:
# PostgreSQL Script for the Data Engineering & SQL question
script = """

-- Create a composite index on user_id and timestamp for better performance
/* CREATE INDEX idx_txs_user_tstamp
 ON 'flutterwave_synthetic_txs.csv' (user_id, timestamp);
*/

/* Create a view for transaction history */
WITH txn_history AS
        (SELECT
            t1.transaction_id
            ,t1.timestamp::TIMESTAMP AS transaction_time
            ,t1.user_id
            ,SUM(t2.amount_usd) AS total_amount_usd_24h
            ,COUNT(DISTINCT CASE
                                WHEN t2.timestamp::TIMESTAMP >= t1.timestamp::TIMESTAMP - INTERVAL '1 hour' THEN t2.card_country
                                ELSE NULL
                            END) AS distinct_card_countries_1h,
            t1.amount_usd,
            t1.card_country
        FROM
            '/flutterwave_synthetic_txs.csv' t1
        LEFT JOIN '/flutterwave_synthetic_txs.csv' t2
            ON t1.user_id = t2.user_id
            AND t2.timestamp::TIMESTAMP <= t1.timestamp::TIMESTAMP
            AND t2.timestamp::TIMESTAMP >= t1.timestamp::TIMESTAMP - INTERVAL '24 hours'
        GROUP BY t1.transaction_id, t1.timestamp, t1.user_id, t1.amount_usd, t1.card_country
        ORDER BY t1.timestamp
        )
SELECT * FROM txn_history
"""
display(duckdb.sql(script).df())

,transaction_id,transaction_time,user_id,total_amount_usd_24h,distinct_card_countries_1h,amount_usd,card_country
0,460ef966-d7c8-40f8-b087-c2eb2c3181c5,2026-05-01 00:00:39,USR_7223,17.58,1,17.58,NG
1,f9b36ab3-4b00-42b0-8cb6-14ee039bbd15,2026-05-01 00:01:25,USR_4643,18.41,1,18.41,NG
2,41685b8a-5ae8-46dd-b200-1e86241267e1,2026-05-01 00:01:53,USR_1183,55.73,1,55.73,GB
3,500a1f97-d7fb-45dd-97bb-7839f565012c,2026-05-01 00:02:06,USR_4727,176.06,1,176.06,KE
4,b0873b43-3a43-4610-a4da-22640950543f,2026-05-01 00:02:06,USR_1944,240.15,1,240.15,US
...,...,...,...,...,...,...,...
52495,05de12a5-e8e6-420a-8867-5b6de7d7b63d,2026-05-31 00:45:58,USR_3622,88.98,1,88.98,NG
52496,ccba8397-e867-4e7d-be4f-afac0d0da0bc,2026-05-31 00:47:02,USR_9169,13.00,1,13.00,NG
52497,56013953-ca33-44be-82cc-b43eccf9d7eb,2026-05-31 00:49:56,USR_1241,38.68,1,20.29,NG
52498,b2b50114-72cb-45d7-bbb6-66a004d2568a,2026-05-31 00:53:04,USR_9648,41.60,1,19.88,NG


In [21]:
# set up and import pandas, numpy, and scikit-learn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, auc

print('Setup Successful')

def execute_eda(df: pd.DataFrame, title: str = "ED Report") -> None:
    """ performs EDA on the given DataFrame and prints a standard output."""
    print(f"===== {title} =====")
    print("\n-- 1. no. of rows and columns ---")
    print(f"rows: {df.shape[0]} | columns: {df.shape[1]}")

    print("\n-- 2. check for missing values ---")
    missing = df.isnull().sum()
    if missing.sum() == 0:
        print("No missing values found.")
    else:
        print(missing[missing > 0])

    print("\n--3. duplicate rows? ---")
    # check for duplicates using trnsaction_id
    duplicate_txs = df.duplicated(subset=['transaction_id']).sum()
    print(f"Duplicate txns: {duplicate_txs}")

    print("\n-- 4. check for fraud (target variable imbalance) ---")
    print(df['is_fraud'].value_counts())
    print(f"Fraud_pct: {(df['is_fraud'].mean() * 100):.2f}%")

    print("\n-- 5. descriptive stats ---")
    num_cols = df.select_dtypes(include=[np.number]).columns
    if not num_cols.empty:
        print(df[num_cols].describe().T.round(2))

    print("\n-- 6. check for high-cardinal columns ---")
    cat_cols = df.select_dtypes(include=['object', 'category']).columns
    for col in cat_cols:
        print(f"{col}: {df[col].nunique()} unique values")
    print("=" * 35 + "\n")

def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """ performs basic data cleaning on the given DataFrame and returns the cleaned DataFrame."""
    clean_df = df.copy()
    # drop duplicates based on transaction_id
    clean_df = clean_df.drop_duplicates(subset=['transaction_id'], keep='first')

    # convert timestamp to datetime
    clean_df['timestamp'] = pd.to_datetime(
        clean_df['timestamp'], format='mixed', errors='coerce', utc=True

      )

    # clean string cols (strip whitespace)
    for col in clean_df.select_dtypes(include=['object']).columns:
        clean_df[col] = clean_df[col].str.strip()

    return clean_df


Setup Successful


In [22]:
# Load the data
raw_df = pd.read_csv('flutterwave_synthetic_txs.csv')

# run EDA temp
execute_eda(raw_df, title="Raw Data EDA")

# clean raw data
clean_df = clean_data(raw_df)

# verify post-clean state
execute_eda(clean_df, title="Cleaned Data EDA")

# reassign cleaned df to raw df for downstream tasks
raw_df = clean_df.copy()

===== Raw Data EDA =====

-- 1. no. of rows and columns ---
rows: 52500 | columns: 9

-- 2. check for missing values ---
No missing values found.

--3. duplicate rows? ---
Duplicate txns: 2500

-- 4. check for fraud (target variable imbalance) ---
is_fraud
0    51500
1     1000
Name: count, dtype: int64
Fraud_pct: 1.90%

-- 5. descriptive stats ---
              count   mean    std  min    25%    50%    75%     max
amount_usd  52500.0  54.80  49.79  5.0  19.32  39.56  74.03  529.43
is_fraud    52500.0   0.02   0.14  0.0   0.00   0.00   0.00    1.00

-- 6. check for high-cardinal columns ---
transaction_id: 50000 unique values
user_id: 8973 unique values
timestamp: 52208 unique values
card_country: 6 unique values
merchant_category: 5 unique values
device_id: 38338 unique values
dispute_status: 4 unique values

===== Cleaned Data EDA =====

-- 1. no. of rows and columns ---
rows: 50000 | columns: 9

-- 2. check for missing values ---
No missing values found.

--3. duplicate rows? ---
Du

In [23]:
df = raw_df.copy()

# define temp features from the cleaned timestamp coln
df['tx_hour'] = df['timestamp'].dt.hour
df['tx_day_of_week'] = df['timestamp'].dt.dayofweek

# drop high-cardinal rows as they aon't aid tree models
cols_to_drop = ['transaction_id', 'user_id', 'timestamp', 'device_id']
df_model = df.drop(columns=cols_to_drop)

# convert text columns into binary (0/1) columns
categorical_cols = ['card_country', 'merchant_category', 'dispute_status']
df_encoded = pd.get_dummies(df_model, columns=categorical_cols, drop_first=True)

print(f"Original shape: {df.shape}")
print(f"Encoded shape: {df_encoded.shape}")

Original shape: (50000, 11)
Encoded shape: (50000, 16)


In [24]:
# separate features (x) and target (y)
X = df_encoded.drop(columns=['is_fraud'])
y = df_encoded['is_fraud']

# stratify the tran & split the test(80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f"Training Fraud cases: {y_train.sum()}")
print(f"Testing Fraud cases: {y_test.sum()}")

Training Fraud cases: 761
Testing Fraud cases: 190


In [25]:
# train the model
# class_weight='balanced' is the silver bullet for imbalanced fraud data
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# generate predictions
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1] # gets the raw probability scores

# evaluate metrics
print("--- Confusion Matrix ---")
# format: [[True Negatives, False Positives], [False Negatives, True Positives]]
print(confusion_matrix(y_test, y_pred))

print("\n--- Classification Report ---")
# focus on the precision and recall for class 1.0
print(classification_report(y_test, y_pred))

--- Confusion Matrix ---
[[9806    4]
 [  68  122]]

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.99      1.00      1.00      9810
           1       0.97      0.64      0.77       190

    accuracy                           0.99     10000
   macro avg       0.98      0.82      0.88     10000
weighted avg       0.99      0.99      0.99     10000

